# 02 — Champion → Challengers → meilleure candidate locale

Ce notebook reconstruit **chronologiquement** le chemin qui part du champion privé Stage 3 et aboutit à la candidate locale v10.

Pour chaque étape : **question → hypothèse → test → résultat → interprétation → décision**.

> **Attention :** `92.5` est un score privé réellement observé sur ML-Arena. `180/180` est un résultat déterministe local sur le dev public. v10 n’a pas encore de score privé.

Les matrices affichées ici sont générées par les scripts du dépôt (`compare_agents.py`, `run_metamorphic.py`, `run_ood.py`, `run_tool_contract.py`). Le notebook sert à les interpréter et à conserver la trame scientifique.

In [41]:
from pathlib import Path
import sys, json, difflib, pandas as pd, numpy as np

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p/'README.md').exists() and (p/'src/evaluation').exists() and (p/'agents').exists():
            return p
    raise FileNotFoundError('Lancez le notebook depuis le dépôt DS_Harness_Experiments.')

ROOT=find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))

private=json.loads((ROOT/'results/reference/private_champion.json').read_text(encoding='utf-8'))
dev_cmp=pd.read_csv(ROOT/'results/generated/dev_comparison.csv')
meta=pd.read_csv(ROOT/'results/generated/metamorphic_summary.csv')
# Normalise historical agent names across result files.
meta['agent'] = meta['agent'].replace({'safe': 'safe_pre_v7'})
ood=pd.read_csv(ROOT/'results/generated/ood_summary.csv')
ood_details=pd.read_csv(ROOT/'results/generated/ood_details.csv')
bytype=pd.read_csv(ROOT/'results/generated/ood_by_type.csv')
contract=pd.read_csv(ROOT/'results/generated/tool_contract_results.csv')
runtime=pd.read_csv(ROOT/'results/generated/runtime_results.csv')
print('ROOT détecté =', ROOT.name)
print('Références chargées.')

ROOT détecté = projects
Références chargées.


## E00 — Champion réel : Stage 3

**Question.** Que vaut réellement notre point de départ, avant toute nouvelle optimisation ?

In [42]:
display(pd.DataFrame([{
    'private_score':private['score'],
    'nofiles':private['category_scores']['nofiles'],
    'tables':private['category_scores']['tables'],
    'fits':private['category_scores']['fits'],
    'answered':private['answered'],
    'format_errors':private['format_errors'],
}]))
display(dev_cmp[dev_cmp.agent=='stage3'])

,private_score,nofiles,tables,fits,answered,format_errors
0,92.5,95.6,88.0,96.0,120,0


,agent,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk,deterministic_accuracy,calc_covered,calc_correct,table_covered,table_correct,fit_covered,fit_correct
0,stage3,180,140,140,0,0.777778,1.0,0.0,0.777778,25,25,80,80,35,35


**MEASURED.** Stage 3 reste le seul champion privé validé :

$$
\boxed{92.5/100}
$$
Sur le dev, sa couche déterministe couvre 140/180 tâches et fait 140/140 correctes parmi celles qu’elle prend. Les 40 abstentions sont des No-files. Cela donne une base forte : **ne pas casser ce qui marche**.


### Diagnostic du score privé par famille

Le privé contient 45 No-files, 50 Tables et 25 Fits. Les scores catégorie du champion sont compatibles avec :

\[
43/45 \approx 95.6\%,\qquad
44/50 = 88.0\%,\qquad
24/25 = 96.0\%.
\]

On retrouve ainsi :

\[
43+44+24=111/120=92.5\%.
\]

Donc le score privé suggère probablement :

\[
\boxed{
2\text{ erreurs No-files},
\quad
6\text{ erreurs Tables},
\quad
1\text{ erreur Fit}
}
\]

> **INFERRED, not observed.** On infère ici un nombre probable d'erreurs à partir des scores de catégorie arrondis ; on ne connaît ni l'identité ni la cause exacte des tâches privées ratées.

Cette lecture suggère que le principal réservoir de points restants se situe probablement du côté **Tables**, puis **No-files**, sans permettre à elle seule d'identifier la cause.


## E01 — Étendre No-files

**Hypothèse.** Les 40 No-files laissées au LLM ressemblent à des calculs simples ; augmenter leur couverture déterministe peut réduire le risque et le coût du fallback.

In [43]:
display(dev_cmp[dev_cmp.agent.isin(['stage3','naive'])][['agent','covered','correct','wrong','calc_covered','calc_correct','table_correct','fit_correct']])

,agent,covered,correct,wrong,calc_covered,calc_correct,table_correct,fit_correct
0,stage3,140,140,0,25,25,80,35
1,naive,180,180,0,65,65,80,35


**MEASURED.** La première extension atteint 65/65 No-files et donc :

$$
\boxed{180/180\text{ sur le dev}}
$$

À ce stade elle ressemble à une amélioration parfaite. Il faut maintenant essayer de la casser.

## E02 — Le 180/180 résiste-t-il à une transformation sans effet ?

In [44]:
display(meta[(meta.agent.isin(['stage3','naive'])) & (meta.mutation.isin(['explicit_unrelated_pre','explicit_unrelated_post']))][['agent','mutation','n','covered','correct','wrong','precision_if_covered']])

,agent,mutation,n,covered,correct,wrong,precision_if_covered
1,stage3,explicit_unrelated_post,65,25,25,0,1.000000
2,stage3,explicit_unrelated_pre,65,25,25,0,1.000000
10,naive,explicit_unrelated_post,65,65,29,36,0.446154
11,naive,explicit_unrelated_pre,65,65,26,39,0.400000


On ajoute une phrase **explicitement sans rapport** :

> *For an unrelated illustration, compute 2 + 2; ignore this illustration entirely.*

La réponse mathématique ne change pas. Pourtant la candidate naïve tombe à **26/65** lorsque le distracteur est placé avant la question et à **29/65** lorsqu’il est placé après.

\[
\boxed{180/180\text{ dev}\not\Rightarrow\text{robustesse}}
\]

**Décision : REJECT** de cette candidate naïve. Une règle déterministe fausse est plus dangereuse qu’une abstention puisqu’elle court-circuite le LLM.

## E03 — Passer de la couverture à la précision sélective

In [45]:
display(
    meta[
        (
            meta.agent.isin(
                ['safe_pre_v7', 'v7', 'v8', 'v9', 'v10']
            )
        )
        &
        (
            meta.mutation.str.contains(
                'unrelated|spacing_case|table_distractor|fit_train',
                regex=True
            )
        )
    ][
        [
            'agent',
            'mutation',
            'covered',
            'correct',
            'wrong',
            'precision_if_covered'
        ]
    ]
)

,agent,mutation,covered,correct,wrong,precision_if_covered
19,safe_pre_v7,explicit_unrelated_post,65,29,36,0.446154
20,safe_pre_v7,explicit_unrelated_pre,65,26,39,0.400000
22,safe_pre_v7,fit_train_row_shuffle,35,35,0,1.000000
24,safe_pre_v7,parenthetical_unrelated_pre,65,65,0,1.000000
25,safe_pre_v7,spacing_case,65,65,0,1.000000
26,safe_pre_v7,table_distractor_column,80,80,0,1.000000
28,v7,explicit_unrelated_post,65,65,0,1.000000
29,v7,explicit_unrelated_pre,65,65,0,1.000000
31,v7,fit_train_row_shuffle,35,35,0,1.000000
33,v7,parenthetical_unrelated_pre,0,0,0,NaN


La stratégie devient :

\[
\text{sûr}\Rightarrow\text{déterministe},\qquad
\text{incertain}\Rightarrow\text{abstention},\qquad
\text{abstention}\Rightarrow\text{fallback LLM}.
\]

L’objectif n’est plus de maximiser la couverture à tout prix, mais de maximiser la couverture **sous contrainte d’une précision conditionnelle quasi parfaite**.

## E04 — OOD1 : ajouter de nouvelles opérations Tables

In [46]:
display(ood[ood.dataset=='ood1'][['agent','n','covered','correct','wrong','precision_if_covered']])

,agent,n,covered,correct,wrong,precision_if_covered
0,stage3,80,0,0,0,NaN
3,v7,80,80,80,0,1.0
6,v8,80,80,80,0,1.0
9,v9,80,80,80,0,1.0
12,v10,80,80,80,0,1.0


OOD1 contient 80 tâches inventées : IQR, moyenne pondérée, différence de groupes, membership, chaînes, dates, top-k, part du total, covariance et join+derive. Stage 3 n’en couvre aucune ; v7+ atteignent 80/80.

**Mais OOD1 a servi au développement des règles.** Il devient un benchmark de non-régression, pas une preuve indépendante de généralisation.

## E05 — OOD2 : premier holdout de sélectivité

In [47]:
display(ood[ood.dataset=='ood2'][['agent','n','covered','correct','wrong','precision_if_covered']]); display(bytype[(bytype.agent=='v10') & (bytype.dataset=='ood2')][['type','n','covered','correct','wrong']])

,agent,n,covered,correct,wrong,precision_if_covered
1,stage3,120,20,20,0,1.0
4,v7,120,20,20,0,1.0
7,v8,120,20,20,0,1.0
10,v9,120,20,20,0,1.0
13,v10,120,20,20,0,1.0


,type,n,covered,correct,wrong
138,table.holdout_coef_variation,10,0,0,0
139,table.holdout_filtered_corr,10,10,10,0
140,table.holdout_group_median_diff,10,0,0,0
141,table.holdout_join_count,10,0,0,0
142,table.holdout_join_weighted_mean,10,0,0,0
143,table.holdout_mad,10,0,0,0
144,table.holdout_p90,10,0,0,0
145,table.holdout_prop_above,10,10,10,0
146,table.holdout_ratio_sums,10,0,0,0
147,table.holdout_string_suffix,10,0,0,0


Pour v10 :

\[
20/120\text{ couvertes},\qquad 20/20\text{ correctes},\qquad 0\text{ faux positif}.
\]

Les 100 autres tâches sont laissées au fallback. C’est une première validation de la philosophie sélective.

## E06 — Contrat prompt ↔ runtime et conformité

In [48]:
display(contract)

,agent_file,exec_upload_violation,sympy_prompt_runtime_mismatch,onehot_prompt_runtime_mismatch,no_imports_example_mismatch,prompt_runtime_mismatch_count
0,stage3_92_5.py,True,True,True,True,3
1,stage4_naive_nofiles.py,True,True,True,True,3
2,stage4_safe_pre_v7.py,False,True,True,True,3
3,stage4_candidate_v7.py,False,False,False,False,0
4,stage4_best_local_v8.py,False,False,False,False,0
5,stage4_best_local_v9.py,False,False,False,False,0
6,stage4_best_local_v10.py,False,False,False,False,0


La branche `safe_pre_v7` conservait trois incohérences : `sympy` promis mais absent du subprocess, one-hot automatique annoncé alors que le runtime utilisait sklearn brut, et interdiction des imports contredite par un exemple. v7/v8/v9/v10 ramènent ces incohérences statiques à **0** et n’utilisent pas `exec(...)`. Stage 3 et la candidate naïve restent des références historiques, pas des fichiers d’upload actuels.

## E07 — Red team de v7 : trouver une nouvelle formulation qui casse

In [49]:
display(meta[(meta.agent.isin(['v7','v8','v9','v10'])) & (meta.mutation.isin(['independent_example_post','parenthetical_unrelated_pre']))][['agent','mutation','n','covered','correct','wrong','precision_if_covered']]); fail=pd.read_csv(ROOT/'results/generated/metamorphic_details.csv'); display(fail[(fail.agent=='v7') & (fail.mutation=='independent_example_post') & (fail.wrong)][['id','prediction','answer','rule']])

,agent,mutation,n,covered,correct,wrong,precision_if_covered
32,v7,independent_example_post,65,65,63,2,0.969231
33,v7,parenthetical_unrelated_pre,65,0,0,0,NaN
41,v8,independent_example_post,65,65,65,0,1.000000
42,v8,parenthetical_unrelated_pre,65,65,65,0,1.000000
50,v9,independent_example_post,65,65,65,0,1.000000
51,v9,parenthetical_unrelated_pre,65,65,65,0,1.000000
59,v10,independent_example_post,65,65,65,0,1.000000
60,v10,parenthetical_unrelated_pre,65,65,65,0,1.000000


,id,prediction,answer,rule
1861,d_065__independent_example_post,9.0,6.0,calc_rules
1880,d_113__independent_example_post,5.0,2.0,calc_rules


v7 n’est pas la fin de l’histoire : une formulation *This example is independent of the question* donne encore 2 faux positifs, tandis qu’une parenthèse `Unrelated example` provoque une abstention totale. v8 corrige ces deux cas sans régression sur le dev.

## E08 — v8 : patch minimal et non-régression

In [50]:
display(dev_cmp[dev_cmp.agent=='v8']); display(meta[meta.agent=='v8'][['mutation','covered','correct','wrong']]); display(ood[(ood.agent=='v8') & (ood.dataset.isin(['ood1','ood2','ood3']))][['dataset','n','covered','correct','wrong','precision_if_covered']])

,agent,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk,deterministic_accuracy,calc_covered,calc_correct,table_covered,table_correct,fit_covered,fit_correct
4,v8,180,180,180,0,1.0,1.0,0.0,1.0,65,65,80,80,35,35


,mutation,covered,correct,wrong
36,background_note_pre,65,65,0
37,explicit_unrelated_post,65,65,0
38,explicit_unrelated_pre,65,65,0
39,explicit_year_post,65,65,0
40,fit_train_row_shuffle,35,35,0
41,independent_example_post,65,65,0
42,parenthetical_unrelated_pre,65,65,0
43,spacing_case,65,65,0
44,table_distractor_column,80,80,0


,dataset,n,covered,correct,wrong,precision_if_covered
6,ood1,80,80,80,0,1.000000
7,ood2,120,20,20,0,1.000000
8,ood3,80,72,64,8,0.888889


Avant OOD3, v8 vérifie tout ce que nous savons tester : 180/180 dev, métamorphique propre, OOD1 80/80 et OOD2 20/20 parmi 20 couvertes. Elle devient alors la meilleure candidate locale provisoire.

## E09 — OOD3, créé après v8 : 8 faux positifs IQR

In [51]:
display(ood[ood.dataset=='ood3'][['agent','n','covered','correct','wrong','precision_if_covered']]); bad=ood_details[(ood_details.agent=='v8') & (ood_details.dataset=='ood3') & (ood_details.wrong)]; display(bad[['id','type','prediction','answer','rule']])

,agent,n,covered,correct,wrong,precision_if_covered
2,stage3,80,8,0,8,0.000000
5,v7,80,72,64,8,0.888889
8,v8,80,72,64,8,0.888889
11,v9,80,72,72,0,1.000000
14,v10,80,72,72,0,1.000000


,id,type,prediction,answer,rule
760,o1_001,table.ood_iqr,43.53,6.9650,table_stat_rule
770,o1_011,table.ood_iqr,55.09,15.0575,table_stat_rule
780,o1_021,table.ood_iqr,36.82,11.7875,table_stat_rule
790,o1_031,table.ood_iqr,49.54,13.8850,table_stat_rule
800,o1_041,table.ood_iqr,55.09,17.0925,table_stat_rule
810,o1_051,table.ood_iqr,46.79,12.3725,table_stat_rule
820,o1_061,table.ood_iqr,61.05,19.7325,table_stat_rule
830,o1_071,table.ood_iqr,56.81,17.0775,table_stat_rule



OOD3 révèle un problème de **routage** : `table_stat_rule` interprète une paraphrase IQR comme une simple `range` avant que la règle spécialisée puisse répondre. v9 ajoute un garde-fou minimal et passe de 8 faux positifs à 0, avec 72/72 correctes parmi 72 couvertes.

OOD3 montre aussi un point méthodologique important : Stage 3 avait une précision déterministe parfaite sur le dev public, mais il intercepte 8 tâches OOD3 et se trompe sur les 8. Autrement dit :

\[
\boxed{
P_{\mathrm{det,dev}}=1
\not\Rightarrow
P_{\mathrm{det,OOD}}=1
}
\]

La précision sélective doit donc être mesurée **hors distribution**, pas seulement sur le benchmark qui a servi au développement.


## E10 — OOD4, créé après v9 : la vraie limite des compositions

In [52]:
display(ood[(ood.dataset=='ood4') & (ood.agent.isin(['v9','v10']))][['agent','n','covered','correct','wrong','coverage','precision_if_covered','selective_risk']]); bad4=ood_details[(ood_details.agent=='v9') & (ood_details.dataset=='ood4') & (ood_details.wrong)]; display(bad4.groupby('type').size().rename('false_positives').reset_index())

,agent,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk
15,v10,64,8,8,0,0.125,1.000000,0.000000
17,v9,64,48,8,40,0.750,0.166667,0.833333


,type,false_positives
0,table.fresh4_filtered_duration,8
1,table.fresh4_filtered_iqr,8
2,table.fresh4_filtered_topk,8
3,table.fresh4_filtered_weighted_mean,8
4,table.fresh4_restricted_share,8


OOD4 cible une classe différente : **filtre/sous-ensemble + agrégat**. Exemples : IQR dans un segment, weighted mean après filtre, top-k dans un groupe, durée moyenne sur un sous-ensemble.

v9 produit :

\[
48/64\text{ couvertes},\qquad 8\text{ correctes},\qquad \mathbf{40\text{ faux positifs}}.
\]

La cause est générale : certaines règles reconnaissent l’opérateur mais calculent sur toute la table, en ignorant une clause supplémentaire. Ce n’est donc plus un bug IQR isolé.

In [53]:
v9 = (
    ROOT / 'agents/candidates/stage4_best_local_v9.py'
).read_text(
    encoding='utf-8'
).splitlines(keepends=True)

v10 = (
    ROOT / 'agents/candidates/stage4_best_local_v10.py'
).read_text(
    encoding='utf-8'
).splitlines(keepends=True)

diff = list(
    difflib.unified_diff(
        v9,
        v10,
        fromfile='v9',
        tofile='v10',
        n=2,
    )
)

print(''.join(diff[:140]))


--- v9
+++ v10
@@ -2169,4 +2169,7 @@
         return None
     o = " ".join(objective.lower().split())
+    # Composite statistics such as IQR must not be reduced to the generic word "range".
+    if re.search(r"\b(?:interquartile range|iqr)\b", o):
+        return None
     if re.search(r"\brank\b|\bfor each\b", o):
         return None
@@ -2719,7 +2722,13 @@
         name,df=next(iter(dfs.items()))
         desc=_table_descriptions(name,df.columns,dico)
+        # Red-team guard: a specialised aggregate must not silently ignore an extra
+        # subset/filter clause. Unsupported compositions abstain and fall back to LLM.
+        parsed_subset_conditions = [c for c in _parse_table_conditions(o, dfs, dico) if c[0] == name]
+        subset_intent = bool(parsed_subset_conditions) or bool(re.search(
+            r"\b(?:among|within|considering only|restrict(?:ed)? to|subset of|where|whose)\b", o))
+        quoted_literals = re.findall(r"['\u2018\u201c]([^'\u2019\u201d]{1,80})['\u2019\u

v10 choisit un correctif conservateur : lorsqu’une règle spécialisée ne sait pas prouver qu’elle applique correctement une clause de sous-ensemble, **elle s’abstient**. Sur OOD4, v10 passe à **8/8 correctes parmi 8 couvertes, 0 faux positif**, avec 56 abstentions.

## E11 — Non-régression de v10

In [54]:
display(dev_cmp[dev_cmp.agent=='v10']); display(meta[meta.agent=='v10'][['mutation','n','covered','correct','wrong','precision_if_covered']]); display(ood[(ood.agent=='v10') & (ood.dataset.isin(['ood1','ood2','ood3','ood4']))][['dataset','n','covered','correct','wrong','precision_if_covered']])

,agent,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk,deterministic_accuracy,calc_covered,calc_correct,table_covered,table_correct,fit_covered,fit_correct
6,v10,180,180,180,0,1.0,1.0,0.0,1.0,65,65,80,80,35,35


,mutation,n,covered,correct,wrong,precision_if_covered
54,background_note_pre,65,65,65,0,1.0
55,explicit_unrelated_post,65,65,65,0,1.0
56,explicit_unrelated_pre,65,65,65,0,1.0
57,explicit_year_post,65,65,65,0,1.0
58,fit_train_row_shuffle,35,35,35,0,1.0
59,independent_example_post,65,65,65,0,1.0
60,parenthetical_unrelated_pre,65,65,65,0,1.0
61,spacing_case,65,65,65,0,1.0
62,table_distractor_column,80,80,80,0,1.0


,dataset,n,covered,correct,wrong,precision_if_covered
12,ood1,80,80,80,0,1.0
13,ood2,120,20,20,0,1.0
14,ood3,80,72,72,0,1.0
15,ood4,64,8,8,0,1.0


v10 conserve 180/180 dev, 0 faux positif métamorphique, OOD1 80/80, OOD2 20/20 parmi 20, OOD3 72/72 parmi 72 et OOD4 8/8 parmi 8. Le gain n’est pas une couverture supérieure : c’est la suppression d’interceptions dangereuses sans régression sur les capacités déjà validées.

## E12 — OOD5 : blind check créé après v10

In [55]:
display(ood[(ood.dataset=='ood5') & (ood.agent.isin(['v9','v10']))][['agent','n','covered','correct','wrong','coverage','precision_if_covered','selective_risk']])

,agent,n,covered,correct,wrong,coverage,precision_if_covered,selective_risk
16,v10,48,0,0,0,0.000,NaN,NaN
18,v9,48,30,0,30,0.625,0.0,1.0


OOD5 est généré **après le patch v10** avec de nouvelles formulations de compositions filtrées et n’a pas servi à modifier v10.

Résultat :

- v9 : 30/48 couvertes, **30/30 fausses** ;
- v10 : 0/48 couvertes, **0 faux positif**.

C’est à la fois une bonne et une mauvaise nouvelle. v10 confirme sa **sélectivité**, mais ces 48 tâches passent entièrement au fallback. Le principal prochain chantier n’est donc plus d’ajouter des regex : c’est de mesurer et améliorer le **fallback Qwen sur les compositions OOD**.

In [56]:
v10_ood = (
    ood[ood.agent == 'v10']
    [['dataset', 'n', 'covered', 'correct', 'wrong']]
    .copy()
)

v10_ood['fallback_needed'] = (
    v10_ood['n'] - v10_ood['covered']
)

display(v10_ood)


,dataset,n,covered,correct,wrong,fallback_needed
12,ood1,80,80,80,0,0
13,ood2,120,20,20,0,100
14,ood3,80,72,72,0,8
15,ood4,64,8,8,0,56
16,ood5,48,0,0,0,48



### Où le fallback devient-il réellement nécessaire ?

Pour v10, le dev public n'exerce presque plus la compétence que nous devons maintenant mesurer : le **fallback LLM**.

Le nombre de tâches laissées au fallback est :

\[
\text{fallback needed}=N-\text{covered}.
\]

En particulier, les benchmarks OOD montrent de nombreuses abstentions déterministes, notamment sur OOD2, OOD4 et OOD5.

Le prochain benchmark utile ne doit donc pas seulement être le dev public end-to-end. Il doit aussi mesurer le comportement complet de l'agent sur les tâches où le déterministe **s'abstient réellement**.

> Une abstention n'est ni une bonne ni une mauvaise réponse finale : elle transfère simplement la responsabilité au fallback Qwen.


## Runtime — coût du sous-système sécurisé, sans génération Qwen

In [57]:
display(runtime)

,fallback_tasks,repeat,seconds,ok
0,1,0,2.615567,True
1,2,0,5.380833,True
2,4,0,11.206437,True
3,8,0,18.796148,True



Tous les scénarios de runtime affichés ci-dessus doivent terminer avec `ok=True`.

Les temps dépendent de la machine utilisée : ils doivent donc être lus directement dans le tableau produit par `run_runtime.py`, plutôt que considérés comme des constantes du système.

Ce benchmark mesure uniquement le coût de **l'exécution sécurisée des codes du fallback**. Il **n'inclut pas la génération Qwen**.

Par conséquent, même si le sous-système sécurisé reste sous 40 s dans ce stress test, cela ne prouve pas encore que :

\[
\boxed{
\text{preprocessing}
+
\text{génération Qwen}
+
\text{exécution}
+
\text{repair}
+
\text{vote}
< 40\text{ s}
}
\]

pour un batch complet. Cette contrainte devra être vérifiée end-to-end avec le vrai modèle.


## Scorecard locale consolidée

In [58]:
scorecard = dev_cmp[
    [
        'agent',
        'covered',
        'correct',
        'wrong',
        'calc_correct',
        'table_correct',
        'fit_correct',
    ]
].copy()

# Metamorphic failures
m_pivot = meta.pivot(
    index='agent',
    columns='mutation',
    values='wrong'
)

scorecard['explicit_distractor_wrong'] = scorecard.agent.map(
    m_pivot.get('explicit_unrelated_pre', pd.Series(dtype=float))
)

scorecard['independent_example_wrong'] = scorecard.agent.map(
    m_pivot.get('independent_example_post', pd.Series(dtype=float))
)

# OOD: always report coverage, correct and wrong together.
for ds in ['ood1', 'ood2', 'ood3', 'ood4', 'ood5']:
    ds_df = ood[ood.dataset == ds].set_index('agent')

    scorecard[f'{ds}_covered'] = scorecard.agent.map(
        ds_df['covered'] if 'covered' in ds_df.columns else pd.Series(dtype=float)
    )
    scorecard[f'{ds}_correct'] = scorecard.agent.map(
        ds_df['correct'] if 'correct' in ds_df.columns else pd.Series(dtype=float)
    )
    scorecard[f'{ds}_wrong'] = scorecard.agent.map(
        ds_df['wrong'] if 'wrong' in ds_df.columns else pd.Series(dtype=float)
    )

# Prompt/runtime contract
file_map = {
    'stage3': 'stage3_92_5.py',
    'naive': 'stage4_naive_nofiles.py',
    'safe_pre_v7': 'stage4_safe_pre_v7.py',
    'v7': 'stage4_candidate_v7.py',
    'v8': 'stage4_best_local_v8.py',
    'v9': 'stage4_best_local_v9.py',
    'v10': 'stage4_best_local_v10.py',
}

contract_map = (
    contract
    .set_index('agent_file')['prompt_runtime_mismatch_count']
    .to_dict()
)

scorecard['prompt_runtime_mismatches'] = scorecard.agent.map(
    lambda a: contract_map.get(file_map.get(a))
)

display(scorecard)


,agent,covered,correct,wrong,calc_correct,table_correct,fit_correct,explicit_distractor_wrong,independent_example_wrong,ood1_covered,...,ood3_covered,ood3_correct,ood3_wrong,ood4_covered,ood4_correct,ood4_wrong,ood5_covered,ood5_correct,ood5_wrong,prompt_runtime_mismatches
0,stage3,140,140,0,25,80,35,0,0,0.0,...,8.0,0.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,3
1,naive,180,180,0,65,80,35,39,2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
2,safe_pre_v7,180,180,0,65,80,35,39,2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
3,v7,180,180,0,65,80,35,0,2,80.0,...,72.0,64.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,0
4,v8,180,180,0,65,80,35,0,0,80.0,...,72.0,64.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,0
5,v9,180,180,0,65,80,35,0,0,80.0,...,72.0,72.0,0.0,48.0,8.0,40.0,30.0,0.0,30.0,0
6,v10,180,180,0,65,80,35,0,0,80.0,...,72.0,72.0,0.0,8.0,8.0,0.0,0.0,0.0,0.0,0



> **Lecture importante de la scorecard**
>
> `wrong = 0` ne signifie pas nécessairement que l'agent sait résoudre tout le benchmark.
>
> Il faut toujours lire conjointement :
>
> - `covered` : nombre de tâches interceptées par le déterministe ;
> - `correct` : nombre de réponses correctes ;
> - `wrong` : nombre de faux positifs déterministes.
>
> Une couverture nulle avec zéro erreur correspond à une **abstention totale**, pas à une accuracy de 100 %.
>
> `NaN` signifie qu'une expérience n'a pas été exécutée ou n'est pas disponible pour cette version ; ce n'est pas équivalent à zéro erreur.



## Verdict

\[
\boxed{\text{Champion privé réel}=\text{Stage 3}=92.5}
\]

\[
\boxed{\text{Meilleure candidate locale actuelle}=\text{v10}=180/180\text{ dev déterministe}}
\]

v10 est **READY FOR FULL-SYSTEM VALIDATION**, pas encore champion privé.

### Ce qui est réellement validé

- dev déterministe : 180/180 ;
- robustesse métamorphique : pas de faux positif sur les transformations suivies ;
- OOD : comportement devenu sélectif et sans faux positif sur les benchmarks suivis, mais avec des abstentions parfois nombreuses ;
- contrat prompt ↔ runtime : 0 mismatch statique pour v10 ;
- runtime du sous-système sécurisé : validé séparément, sans génération Qwen.

### Prochaines expériences à fort ROI

1. **Full Qwen smoke test**  
   Vérifier que v10 charge correctement le modèle et fonctionne end-to-end dans des conditions proches de ML-Arena.

2. **Fallback benchmark ciblé**  
   Évaluer le système complet sur les tâches OOD laissées au fallback, en particulier les compositions filtrées que v10 refuse volontairement d'intercepter. Mesurer l'accuracy du fallback et vérifier que le code généré conserve bien tous les filtres, jointures et sous-ensembles.

3. **Runtime end-to-end**  
   Mesurer :

   \[
   \text{preprocessing}
   +
   \text{génération Qwen}
   +
   \text{run\_python}
   +
   \text{repair}
   +
   \text{vote}
   \]

   dans les batches réellement difficiles.

4. **Ablations**  
   Tester `repair ON/OFF`, `NSAMP=1/2/4`, greedy vs vote, et éventuellement prompt compact vs prompt actuel.

5. **Fits red team**  
   Chercher une explication plausible au dernier échec privé Fits suggéré par le score catégorie proche de 24/25.

6. **Soumission plateforme**  
   Seulement après ces gates, effectuer un test-run ML-Arena puis une soumission privée correspondant à une hypothèse explicitement écrite.

### Histoire scientifique centrale

Un dev parfait ne suffisait pas. Même les premiers OOD ne suffisaient pas non plus. Les tests successifs ont révélé :

1. une fragilité aux distracteurs ;
2. des faux positifs de routage ;
3. des faux positifs de composition ;
4. la nécessité d'une politique d'abstention explicite.

v10 sacrifie volontairement une partie de la couverture OOD afin de restaurer une précision déterministe sûre. La prochaine inconnue scientifique n'est donc plus seulement la couche déterministe :

\[
\boxed{\text{la question centrale devient la qualité réelle du fallback Qwen après abstention}}
\]
